In [ ]:
import tensorflow as tf
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random



from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

In [ ]:
!nvidia-smi

In [ ]:
image_dir = "/kaggle/input/datasets/sakshaymahna/kittiroadsegmentation/training/image_2"
mask_dir  = "/kaggle/input/datasets/sakshaymahna/kittiroadsegmentation/training/gt_image_2"

images = os.listdir(image_dir)

print("Total images:", len(images))

In [ ]:
sample = images[10]

img = cv2.imread(os.path.join(image_dir, sample))
print("Image shape:", img.shape)

plt.imshow(img)

In [ ]:
plt.figure(figsize=(15,10))

for i in range(6):

    sample = random.choice(images)

    prefix, rest = sample.split("_",1)
    mask_name = f"{prefix}_road_{rest}"

    img = cv2.imread(os.path.join(image_dir, sample))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(os.path.join(mask_dir, mask_name))
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    plt.subplot(3,4,i*2+1)
    plt.imshow(img)
    plt.title("Image")

    plt.subplot(3,4,i*2+2)
    plt.imshow(mask)
    plt.title("Mask")

plt.show()

In [ ]:
pixels = []

for img_name in images[:50]:
    img = cv2.imread(os.path.join(image_dir, img_name))
    pixels.append(img.mean())

print("Mean pixel intensity:", np.mean(pixels))

In [ ]:
plt.hist(pixels, bins=20)
plt.title("Image brightness distribution")
plt.show()

In [ ]:
for img_name in images[:5]:

    prefix, rest = img_name.split("_",1)
    mask_name = f"{prefix}_road_{rest}"

    print(img_name, "->", mask_name)

In [ ]:
heatmap = np.zeros((375,1242))

for img_name in images[:50]:

    prefix, rest = img_name.split("_",1)
    mask_name = f"{prefix}_road_{rest}"

    mask = cv2.imread(os.path.join(mask_dir, mask_name))

    road = mask[:,:,0] == 255

    road = cv2.resize(road.astype(np.uint8), (1242,375))

    heatmap += road

plt.imshow(heatmap, cmap="hot")
plt.title("Road pixel frequency")
plt.colorbar()
plt.show()

In [ ]:
def get_mask_name(img_name):
    prefix, rest = img_name.split("_",1)
    return f"{prefix}_road_{rest}"

In [ ]:
IMG_SIZE = 256

def load_sample(img_name):

    img_path = os.path.join(image_dir, img_name)
    mask_path = os.path.join(mask_dir, get_mask_name(img_name))

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(mask_path)

    # road pixels
    road = mask[:,:,0] == 255
    mask = road.astype(np.float32)

    img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
    mask = cv2.resize(mask,(IMG_SIZE,IMG_SIZE))

    mask = np.expand_dims(mask,axis=-1)

    return img, mask

In [ ]:
X=[]
Y=[]

for img_name in images:
    
    img,mask = load_sample(img_name)

    X.append(img)
    Y.append(mask)

X = np.array(X)/255.0
Y = np.array(Y)

print(X.shape,Y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_val,Y_train,Y_val = train_test_split(
    X,Y,test_size=0.2,random_state=42
)

In [ ]:
def build_resnet_unet():

    base = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # Encoder layers (skip connections)
    s1 = base.get_layer("conv1_relu").output
    s2 = base.get_layer("conv2_block3_out").output
    s3 = base.get_layer("conv3_block4_out").output
    s4 = base.get_layer("conv4_block6_out").output

    b1 = base.get_layer("conv5_block3_out").output

    # Decoder
    d1 = UpSampling2D((2,2))(b1)
    d1 = Concatenate()([d1,s4])
    d1 = Conv2D(512,3,padding="same",activation="relu")(d1)

    d2 = UpSampling2D((2,2))(d1)
    d2 = Concatenate()([d2,s3])
    d2 = Conv2D(256,3,padding="same",activation="relu")(d2)

    d3 = UpSampling2D((2,2))(d2)
    d3 = Concatenate()([d3,s2])
    d3 = Conv2D(128,3,padding="same",activation="relu")(d3)

    d4 = UpSampling2D((2,2))(d3)
    d4 = Concatenate()([d4,s1])
    d4 = Conv2D(64,3,padding="same",activation="relu")(d4)

    d5 = UpSampling2D((2,2))(d4)
    outputs = Conv2D(1,1,activation="sigmoid")(d5)

    model = Model(base.input,outputs)

    return model

In [ ]:
model = build_resnet_unet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val,Y_val),
    epochs=40,
    batch_size=8
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['accuracy'], label="Train Accuracy")
plt.plot(history.history['val_accuracy'], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label="Train Loss")
plt.plot(history.history['val_loss'], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.show()

In [ ]:
import tarfile, os

model.export("/kaggle/working/road_seg_savedmodel")

with tarfile.open("/kaggle/working/model.tar.gz", "w:gz") as tar:
    tar.add("/kaggle/working/road_seg_savedmodel", arcname=".")

In [ ]:
idx = 4
pred = model.predict(X_val[idx:idx+1])

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(X_val[idx])
plt.title("Image")

plt.subplot(1,3,2)
plt.imshow(Y_val[idx].squeeze())
plt.title("True Mask")

plt.subplot(1,3,3)
plt.imshow(pred[0].squeeze() > 0.5)
plt.title("Prediction")

plt.show()

In [ ]:
video_path = "/kaggle/input/datasets/sakshaymahna/kittiroadsegmentation/testing/solidYellowLeft.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

print(width, height, fps)

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter("/kaggle/working/output.mp4",fourcc,fps,(width, height))

In [ ]:
IMG_SIZE = 256

def process_frame(frame):

    original = frame.copy()

    small = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    small = small / 255.0

    pred = model.predict(small[np.newaxis,...])[0]

    mask = pred.squeeze() > 0.5

    mask = cv2.resize(mask.astype(np.uint8), (width, height))

    overlay = original.copy()
    overlay[mask == 1] = [0,255,0]

    return overlay

In [ ]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    result = process_frame(frame)
    out.write(result)
cap.release()
out.release()